In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [2]:
SITES = ["Duliajan", "Moran", "Naharkatiya", "Digboi", "Brahmaputra", "Others"]
ACTIVITIES = ["Maintenance", "Operations", "Construction", "Logistics", "Inspection", "Others"]
PRECURSORS = [
    "Inadequate isolation", "Poor housekeeping",
    "Bypass of safety device", "Inadequate permit-to-work",
    "Lack of PPE", "Improper lifting",
    "Hot work near flammables", "Confined space entry violation",
    "Electrical safety breach", "Vehicle / driving violation",
    "Dropped object", "Scaffolding deficiency",
    "Pressure safety concern", "Fall from height risk",
    "Chemical exposure",
]
NUM_POINTS = 900
MIN_MARKER, MAX_MARKER = 3, 20

In [3]:
# yahan real data wire karna — bas ye function replace karo
# df mein columns chahiye: site, activity, precursor, sif_probability, severity, reports

def generate_random_data(n=NUM_POINTS, seed=None):
    rng = np.random.default_rng(seed)
    sr = {"Duliajan":.08,"Moran":.04,"Naharkatiya":.02,"Digboi":0,"Brahmaputra":-.02,"Others":-.03}
    ar = {"Maintenance":.10,"Operations":.05,"Construction":.03,"Logistics":0,"Inspection":-.03,"Others":-.05}
    sites = rng.choice(SITES, n, p=[.25,.20,.18,.15,.12,.10])
    acts  = rng.choice(ACTIVITIES, n, p=[.28,.22,.18,.14,.10,.08])
    precs = rng.choice(PRECURSORS, n)
    base  = rng.beta(2, 8, n)
    sif   = np.clip((base + np.array([sr[s] for s in sites]) + np.array([ar[a] for a in acts]) + rng.normal(0,.03,n)) * 100, 0, 60)
    sev   = np.clip((sif/60)*8 + 1 + rng.normal(0,1.2,n), 1, 10).astype(int)
    return pd.DataFrame(dict(site=sites, activity=acts, precursor=precs,
                             sif_probability=np.round(sif,1), severity=sev, reports=rng.integers(1,200,n)))

df = generate_random_data()
df.head(6)

,site,activity,precursor,sif_probability,severity,reports
0,Moran,Maintenance,Confined space entry violation,25.8,4,18
1,Duliajan,Operations,Chemical exposure,47.2,6,128
2,Digboi,Construction,Pressure safety concern,46.0,5,112
3,Duliajan,Operations,Inadequate permit-to-work,31.8,5,10
4,Duliajan,Maintenance,Chemical exposure,42.2,5,122
5,Duliajan,Maintenance,Pressure safety concern,23.1,4,39


In [4]:
sm = {s:i for i,s in enumerate(SITES)}
am = {a:i for i,a in enumerate(ACTIVITIES)}
rj = np.random.default_rng()
df["s_j"] = df.site.map(sm) + rj.uniform(-.32,.32,len(df))
df["a_j"] = df.activity.map(am) + rj.uniform(-.32,.32,len(df))
lo, hi = df.severity.min(), df.severity.max()
df["ms"] = MIN_MARKER + (df.severity - lo) / max(hi-lo,1) * (MAX_MARKER - MIN_MARKER)

In [5]:
SIF_MAX = max(55, df.sif_probability.max() + 2)

# cyan se start — sab dots black pe clearly dikhenge
COLORSCALE = [
    [0.00, "rgb(0,  240, 255)"],
    [0.12, "rgb(0,  200, 230)"],
    [0.24, "rgb(0,  220, 160)"],
    [0.36, "rgb(50, 230, 100)"],
    [0.48, "rgb(140, 240, 60)"],
    [0.58, "rgb(220, 240, 30)"],
    [0.68, "rgb(255, 210, 0)"],
    [0.78, "rgb(255, 160, 0)"],
    [0.88, "rgb(255, 95, 10)"],
    [0.95, "rgb(240, 40, 10)"],
    [1.00, "rgb(210, 0,  0)"],
]

BG       = "#000000"
PANEL    = "rgba(8, 12, 24, 0.95)"
GRID     = "rgba(60, 120, 200, 0.28)"
AX_LINE  = "rgba(90, 150, 240, 0.60)"
TK_COL   = "#a8c4f0"
TT_COL   = "#e0eaff"

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=df["a_j"], y=df["sif_probability"], z=df["s_j"],
    mode="markers",
    marker=dict(
        size=df["ms"],
        color=df["sif_probability"],
        colorscale=COLORSCALE,
        cmin=0, cmax=SIF_MAX,
        opacity=0.95,
        line=dict(width=0.5, color="rgba(255,255,255,0.25)"),
        colorbar=dict(
            title=dict(text="SIF Probability (%)",
                       font=dict(size=13, family="Inter, sans-serif", color=TT_COL), side="right"),
            thickness=18, len=0.62, x=1.02, y=0.52,
            tickfont=dict(size=11, family="Inter, sans-serif", color=TK_COL),
            dtick=10, outlinewidth=1, outlinecolor="rgba(100,160,255,0.3)",
            bgcolor="rgba(0,0,0,0.6)",
        ),
    ),
    hovertemplate=(
        '<b>Site:</b> %{customdata[0]}<br>'
        '<b>Activity:</b> %{customdata[1]}<br>'
        '<b>Precursor:</b> %{customdata[2]}<br>'
        '<b>SIF Probability:</b> %{customdata[3]:.0f}%<br>'
        '<b>Severity:</b> %{customdata[5]}/10<br>'
        '<b>Reports:</b> %{customdata[4]}'
        '<extra></extra>'
    ),
    customdata=df[["site","activity","precursor","sif_probability","reports","severity"]].values,
))

def mk_ax(title, tickvals=None, ticktext=None, rng=None, dtick=None, tsuffix=""):
    d = dict(
        title=dict(text=f"<b>{title}</b>", font=dict(size=15, family="Inter, sans-serif", color=TT_COL)),
        tickfont=dict(size=12, family="Inter, sans-serif", color=TK_COL),
        gridcolor=GRID, gridwidth=1,
        showbackground=True, backgroundcolor=PANEL,
        linecolor=AX_LINE, linewidth=2, zerolinecolor=AX_LINE, zerolinewidth=2,
        showline=True, mirror=True,
        showspikes=True, spikesides=True, spikethickness=1, spikecolor="rgba(100,160,255,0.3)",
    )
    if tickvals is not None: d["tickvals"], d["ticktext"] = tickvals, ticktext
    if rng is not None: d["range"] = rng
    if dtick: d["dtick"] = dtick
    if tsuffix: d["ticksuffix"] = tsuffix
    return d

fig.update_layout(
    title=dict(
        text=(
            '<b style="font-size:22px;color:#ffffff;">SIF Precursor Landscape (3D)</b>'
            '<br><span style="font-size:13px;color:#7b9ac8;">'
            'Each point represents a report (size = severity, color = SIF Probability)'
            '</span>'
        ),
        x=0.5, xanchor="center", y=0.965, font=dict(family="Inter, sans-serif"),
    ),
    scene=dict(
        xaxis=mk_ax("Activity",  list(range(len(ACTIVITIES))), ACTIVITIES, [-0.6, len(ACTIVITIES)-0.4]),
        yaxis=mk_ax("SIF Probability (%)", rng=[0, SIF_MAX+2], dtick=10, tsuffix="%"),
        zaxis=mk_ax("Site",      list(range(len(SITES))), SITES, [-0.6, len(SITES)-0.4]),
        camera=dict(eye=dict(x=-1.80, y=1.25, z=0.85), center=dict(x=0, y=-0.08, z=0), up=dict(x=0, y=1, z=0)),
        aspectmode="manual", aspectratio=dict(x=1.4, y=1.0, z=1.1),
        bgcolor=BG,
    ),
    width=1100, height=750,
    margin=dict(l=10, r=40, t=80, b=10),
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family="Inter, sans-serif", color=TT_COL),
    hoverlabel=dict(bgcolor="rgba(10,14,30,0.95)", font_size=13,
                    font_family="Inter, sans-serif", font_color="#e0eaff",
                    bordercolor="rgba(100,160,255,0.5)"),
)

fig.show()

In [6]:
fig.write_html("sif_precursor_landscape_3d.html", include_plotlyjs="cdn", full_html=True,
               config={"displayModeBar":True,"scrollZoom":True,"displaylogo":False,
                       "modeBarButtonsToRemove":["lasso2d","select2d"]})